# 02. Deduplicar Splink + avaliar vs coorte


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    SPLINK_INPUT_VIEW,
    USE_PHONETIC_STRIP_VOWELS,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
require_tables(con, ['registro_unificado', 'ground_truth_clusters'], notebook_origem='00')
materialize_splink_input(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)


In [ ]:
from splink import Linker, SettingsCreator, block_on
import splink.comparison_library as cl

input_cols = set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
db_api = get_splink_db_api(con)
blocking_rules = [
    block_on('substr(primeiro_nome,1,3)', 'substr(ultimo_nome,1,4)'),
    block_on('ultimo_nome', 'data_nascimento'),
    block_on('primeiro_nome', 'data_nascimento'),
    block_on('substr(cep,1,5)', 'primeiro_nome'),
    block_on('nome_mae', 'data_nascimento'),
]
comparisons = [
    cl.NameComparison('nome_completo'),
    cl.NameComparison('primeiro_nome').configure(term_frequency_adjustments=True),
    cl.NameComparison('ultimo_nome').configure(term_frequency_adjustments=True),
    cl.NameComparison('nome_completo_phon'),
    cl.DateOfBirthComparison('data_nascimento', input_is_string=True),
    cl.NameComparison('nome_mae'),
    cl.ExactMatch('sexo').configure(term_frequency_adjustments=True),
    cl.ExactMatch('uf').configure(term_frequency_adjustments=True),
]
if USE_PHONETIC_STRIP_VOWELS and 'nome_completo_phon_sv' in input_cols:
    comparisons.append(cl.NameComparison('nome_completo_phon_sv'))

settings = SettingsCreator(
    link_type='dedupe_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
)
linker = Linker(SPLINK_INPUT_VIEW, settings, db_api=db_api)


In [ ]:
deterministic_rules = [
    block_on('primeiro_nome', 'ultimo_nome', 'data_nascimento'),
    block_on('nome_mae', 'data_nascimento'),
]
linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.8)
linker.training.estimate_u_using_random_sampling(max_pairs=1_000_000)
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome', 'ultimo_nome', 'data_nascimento')
)


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).

In [ ]:
linker.visualisations.match_weights_chart()
linker.evaluation.unlinkables_chart()

## Predict + clustering

In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = df_predict.as_pandas_dataframe()
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())


## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).

In [ ]:
records_sample = df_predict.as_pandas_dataframe(limit=5).to_dict(orient='records')
linker.visualisations.waterfall_chart(records_sample, filter_nulls=False)

## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).

In [ ]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))

## Avaliação vs labels (`cluster`)

Curva accuracy/precision/recall por threshold e waterfalls de erros.

In [ ]:
linker.evaluation.accuracy_analysis_from_labels_column(
    'cluster',
    output_type='accuracy',
    match_weight_round_to_nearest=0.02,
)

In [ ]:
records_fp = linker.evaluation.prediction_errors_from_labels_column(
    'cluster',
    threshold_match_probability=0.999,
    include_false_negatives=False,
    include_false_positives=True,
).as_record_dict()
linker.visualisations.waterfall_chart(records_fp)

In [ ]:
# Parte dos falsos negativos não passou pelas blocking rules
records_fn = linker.evaluation.prediction_errors_from_labels_column(
    'cluster',
    threshold_match_probability=0.5,
    include_false_negatives=True,
    include_false_positives=False,
).as_record_dict(limit=50)
linker.visualisations.waterfall_chart(records_fn)

## Métricas coorte (recall cross-source)

Recall manual: pares GT (`gt_*`) no mesmo cluster Splink.

In [ ]:
pairs_gt = con.execute('''
    SELECT g1.unique_id AS id_censo, g2.unique_id AS id_cpf
    FROM ground_truth_clusters g1
    JOIN ground_truth_clusters g2 ON g1.cluster = g2.cluster
    WHERE g1.unique_id LIKE 'censo_%' AND g2.unique_id LIKE 'cpf_%' AND g1.cluster LIKE 'gt_%'
''').df()

pred_map = df_clusters.set_index('unique_id')['cluster_id'].to_dict()
hits = sum(
    1 for _, row in pairs_gt.iterrows()
    if pred_map.get(row['id_censo']) == pred_map.get(row['id_cpf'])
    and pred_map.get(row['id_censo']) is not None
)
recall = hits / len(pairs_gt) if len(pairs_gt) else 0.0
print(f'Recall cross-source: {hits}/{len(pairs_gt)} = {recall:.3f}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(OUTPUT_DIR / 'splink_predictions.parquet')
df_clusters.to_parquet(OUTPUT_DIR / 'splink_clusters.parquet')
pd.DataFrame([{'recall_cross_source': recall, 'n_pares_gt': len(pairs_gt), 'hits': hits}]).to_csv(
    OUTPUT_DIR / 'metricas_cohort.csv', index=False
)
con.close()
